# Configuration: data and output paths

Paths used by every notebook in this folder.

Edit the three paths below if your folders move. Every other notebook loads this
one first, so this is the only place paths are set. Environment variables with the
same names override these values (useful for testing).

**Run order:** 00_config, then 03 (benchmark), 04 (semi-synthetic), 05 (analysis);
for MIMIC-IV: 06 (extract), then 07 (temporal validation). 01 and 02 are libraries
loaded automatically; 08 makes synthetic MIMIC-IV-shaped files for testing only.

**Python packages:** numpy, pandas, scipy, scikit-learn, xgboost, interpret-core,
shap, lime, pyreadr, matplotlib, duckdb, nbformat (see the install cell below).

## Install packages (run once)
Remove the `#` to install.

In [31]:
# %pip install numpy pandas scipy scikit-learn xgboost interpret-core shap lime pyreadr matplotlib duckdb nbformat ucimlrepo

In [32]:
import os

## Paths

In [33]:
# ---- paths
DATA_DIR = os.environ.get("DATA_DIR", r"D:\New_Research_2027\Clinical_XAI_Faithfulness")
MIMIC_ROOT = os.environ.get("MIMIC_ROOT", r"D:\MIMIC_IV\mimic_iv")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", os.path.join(DATA_DIR, "outputs"))
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")                # open-data benchmark results
MIMIC_COHORT = os.path.join(OUTPUT_DIR, "mimic_cohort.csv.gz")   # extracted MIMIC-IV cohort
MIMIC_RESULTS_DIR = os.path.join(OUTPUT_DIR, "results_mimic")    # MIMIC-IV results (tables, macros)
PAPER_GEN_DIR = os.path.join(OUTPUT_DIR, "paper_generated")      # tables, figures, macros for the LaTeX paper
for _d in (OUTPUT_DIR, RESULTS_DIR, MIMIC_RESULTS_DIR, PAPER_GEN_DIR):
    os.makedirs(_d, exist_ok=True)

## File search

### `find_file` (function)

Return the first file under `root` (searched recursively) whose name matches one of `names`
(case-insensitive, in the order given), or None.

In [34]:
# ---- file search
def find_file(*names, root=None):
    """Return the first file under `root` (searched recursively) whose name matches one of `names`
    (case-insensitive, in the order given), or None."""
    root = root or DATA_DIR
    index = {}
    for dirpath, _, files in os.walk(root):
        if os.path.abspath(dirpath).startswith(os.path.abspath(OUTPUT_DIR)):
            continue
        for f in files:
            index.setdefault(f.lower(), os.path.join(dirpath, f))
    for n in names:
        if n.lower() in index:
            return index[n.lower()]
    return None

### `find_mimic_root` (function)

Return the folder under `root` that contains MIMIC-IV's hosp/ and icu/ sub-folders.

In [35]:
def find_mimic_root(root=None, max_depth=3):
    """Return the folder under `root` that contains MIMIC-IV's hosp/ and icu/ sub-folders."""
    root = os.path.abspath(root or MIMIC_ROOT)
    base_depth = root.rstrip(os.sep).count(os.sep)
    for dirpath, dirnames, _ in os.walk(root):
        names = {d.lower() for d in dirnames}
        if "hosp" in names and "icu" in names:
            return dirpath
        if dirpath.count(os.sep) - base_depth >= max_depth:
            dirnames[:] = []
    raise FileNotFoundError(f"No folder with hosp/ and icu/ found under {root}")

## Download from the UCI Machine Learning Repository (ucimlrepo)

In [36]:
# ---- download from the UCI Machine Learning Repository (ucimlrepo)
UCI_DIR = os.path.join(DATA_DIR, "uci_downloads")
UCI_DATASETS = {  # benchmark name: (UCI id, file saved in UCI_DIR)
    "CKD": (336, "uci_336_chronic_kidney_disease.csv"),
    "Heart": (45, "uci_45_heart_disease.csv"),
    "HeartFailure": (519, "uci_519_heart_failure_clinical_records.csv"),
    "BreastCancer": (17, "uci_17_breast_cancer_wisconsin_diagnostic.csv"),
    "Cirrhosis": (878, "uci_878_cirrhosis_patient_survival.csv"),
    "SUPPORT2": (880, "uci_880_support2.csv"),
    "Diabetes130": (296, "uci_296_diabetes_130_us_hospitals.csv"),
}

### `fetch_uci` (function)

Download one dataset with ucimlrepo.fetch_ucirepo (IDs, features and targets together, i.e.
`.data.original`) and save it as CSV in UCI_DIR. Later runs read the saved file, so the internet
is needed only once. Returns the CSV path.

In [37]:
def fetch_uci(name, force=False):
    """Download one dataset with ucimlrepo.fetch_ucirepo (IDs, features and targets together, i.e.
    `.data.original`) and save it as CSV in UCI_DIR. Later runs read the saved file, so the internet
    is needed only once. Returns the CSV path."""
    uci_id, fname = UCI_DATASETS[name]
    path = os.path.join(UCI_DIR, fname)
    if os.path.exists(path) and not force:
        return path
    from ucimlrepo import fetch_ucirepo  # pip install ucimlrepo
    print(f"[{name}] downloading UCI dataset id={uci_id} ...")
    ds = fetch_ucirepo(id=uci_id)
    df = ds.data.original
    if df is None:
        import pandas as pd
        df = pd.concat([p for p in (ds.data.ids, ds.data.features, ds.data.targets) if p is not None], axis=1)
    os.makedirs(UCI_DIR, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"[{name}] saved {df.shape[0]} rows x {df.shape[1]} columns to {path}")
    return path

### `fetch_all_uci` (function)

Download every benchmark dataset from UCI (skips files already saved).

In [38]:
def fetch_all_uci(force=False):
    """Download every benchmark dataset from UCI (skips files already saved)."""
    return {n: fetch_uci(n, force) for n in UCI_DATASETS}

## Download the datasets from UCI (run once)
Downloads every dataset that is not already in `DATA_DIR` with `ucimlrepo.fetch_ucirepo` and saves it in `DATA_DIR/uci_downloads/`. Needs an internet connection the first time only. Datasets you already have as local files are left alone (the loaders use those first). The loaders also download automatically if a dataset is missing, so this cell is optional.

In [39]:
if __name__ == "__main__":
    for _name in UCI_DATASETS:
        try:
            fetch_uci(_name)
        except Exception as _e:
            print(f"[{_name}] download failed: {type(_e).__name__}: {_e}")

## Check what will be read
Lists the file that will be used for each dataset and the MIMIC-IV folder.

In [40]:
if __name__ == "__main__":
    local_names = {
        "CKD": ("ckd.csv", "kidney_disease.csv", "chronic_kidney_disease_full.arff", "chronic_kidney_disease.arff", "chronic_kidney_disease.csv"),
        "Heart": ("heart_kmed.rda", "heart.rda", "processed.cleveland.data", "heart_cleveland_upload.csv", "cleveland.csv"),
        "HeartFailure": ("hf.csv", "heart_failure_clinical_records_dataset.csv", "heart_failure_clinical_records.csv", "cardiac_failure_df.csv"),
        "BreastCancer": ("wdbc.data", "breast_cancer_wisconsin.csv", "breast-cancer-wisconsin-diagnostic.csv", "wdbc.csv"),
        "Cirrhosis": ("pbc.csv", "cirrhosis.csv"),
        "SUPPORT2": ("support.rda", "support2.csv"),
        "Diabetes130": ("readmission.rda", "diabetic_data.csv"),
    }
    print("DATA_DIR  :", DATA_DIR, "(exists)" if os.path.isdir(DATA_DIR) else "(MISSING)")
    for name, names in local_names.items():
        f = find_file(*names, UCI_DATASETS[name][1])
        print(f"  {name:13s}", f or f"not downloaded yet - will be fetched from UCI (id {UCI_DATASETS[name][0]})")
    try:
        print("MIMIC-IV  :", find_mimic_root())
    except FileNotFoundError as e:
        print("MIMIC-IV  : NOT FOUND -", e)
    print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_DIR  : D:\New_Research_2027\Clinical_XAI_Faithfulness (exists)
  CKD           D:\New_Research_2027\Clinical_XAI_Faithfulness\uci_downloads\uci_336_chronic_kidney_disease.csv
  Heart         D:\New_Research_2027\Clinical_XAI_Faithfulness\heart+disease\processed.cleveland.data
  HeartFailure  D:\New_Research_2027\Clinical_XAI_Faithfulness\heart+failure+clinical+records\heart_failure_clinical_records_dataset.csv
  BreastCancer  D:\New_Research_2027\Clinical_XAI_Faithfulness\breast+cancer+wisconsin+diagnostic\wdbc.data
  Cirrhosis     D:\New_Research_2027\Clinical_XAI_Faithfulness\cirrhosis+patient+survival+prediction+dataset-1\cirrhosis.csv
  SUPPORT2      D:\New_Research_2027\Clinical_XAI_Faithfulness\uci_downloads\uci_880_support2.csv
  Diabetes130   D:\New_Research_2027\Clinical_XAI_Faithfulness\diabetes+130-us+hospitals+for+years+1999-2008\diabetic_data.csv
MIMIC-IV  : D:\MIMIC_IV\mimic_iv\mimic-iv-3.1
OUTPUT_DIR: D:\New_Research_2027\Clinical_XAI_Faithfulness\outputs
